# Advanced Features

This notebook covers advanced features of ScraperNHL:
- Caching
- Rate limiting
- Data transformations
- Error handling
- Performance optimization

In [ ]:
from scrapernhl import scrape, HockeyScraper
from scrapernhl.utils import Cache, RateLimiter
import pandas as pd
import time

## Caching

ScraperNHL automatically caches API responses to improve performance and reduce API load.

### How Caching Works

- Responses are cached based on the URL
- Different TTL (Time To Live) for different data types:
  - Play-by-play: Long cache (games don't change)
  - Stats: Medium cache (updates periodically)
  - Schedule: Short cache (changes frequently)
- Cache is stored in memory during the session

In [ ]:
# Example: See caching in action
scraper = HockeyScraper('qmjhl')

# First request (fetches from API)
print("First request...")
start = time.time()
stats1 = scraper.player_stats(season=90, position='skaters', limit=10)
time1 = time.time() - start
print(f"Time: {time1:.3f} seconds")

# Second request (uses cache)
print("\nSecond request (cached)...")
start = time.time()
stats2 = scraper.player_stats(season=90, position='skaters', limit=10)
time2 = time.time() - start
print(f"Time: {time2:.3f} seconds")

print(f"\nSpeedup: {time1/time2:.1f}x faster")

## Rate Limiting

ScraperNHL automatically rate limits requests to respect API endpoints.

### Default Rate Limits

- **Non-NHL leagues**: 2 requests per second
- **NHL**: No rate limiting (NHL API is more robust)

The rate limiter automatically throttles requests when you exceed the limit.

In [ ]:
# Example: Making multiple requests
scraper = HockeyScraper('ohl')

print("Making 5 requests with rate limiting...")
start = time.time()

for i in range(5):
    print(f"Request {i+1}...", end=" ")
    stats = scraper.player_stats(season=83, position='skaters', limit=5, offset=i*5)
    print(f"Done ({len(stats)} rows)")

elapsed = time.time() - start
print(f"\nTotal time: {elapsed:.2f} seconds")
print(f"Average: {elapsed/5:.2f} seconds per request")

## Data Transformations

ScraperNHL includes a transformation pipeline that normalizes data.

### Transform Pipeline

1. **Extract nested data**: Flattens JSON responses
2. **Normalize columns**: Standardizes column names
3. **Clean values**: Removes null/empty values
4. **Type conversion**: Converts strings to numbers where appropriate

In [ ]:
# Example: Custom transformation
from scrapernhl.transform import transform_pbp, nhlify_goals

# Get play-by-play data
game_id = 31171  # Replace with valid game ID

try:
    # Without transformations
    pbp_raw = scrape('qmjhl', 'pbp', game_id=game_id, nhlify=False)
    
    # Apply nhlify transformation
    pbp_clean = nhlify_goals(pbp_raw)
    
    print(f"Raw: {len(pbp_raw)} events")
    print(f"After nhlify: {len(pbp_clean)} events")
    
    # Show difference
    if 'event_type' in pbp_raw.columns:
        print("\nRaw event types:")
        print(pbp_raw['event_type'].value_counts().head())
        
        print("\nCleaned event types:")
        print(pbp_clean['event_type'].value_counts().head())
except Exception as e:
    print(f"Error: {e}")

### Goal Replay Tracking: `tracking_dict_to_df()`

Convert raw NHL goal-replay sprite frames (from `goal_replay()`) into a
tidy DataFrame with rink coordinates and optional movement metrics.


In [ ]:
from scrapernhl import HockeyScraper, tracking_dict_to_df

nhl = HockeyScraper('nhl')

# Fetch goal replay frames from a known replay URL
frames = nhl.goal_replay(
    'https://wsr.nhle.com/sprites/20232024/2023020001/ev154.json'
)

# Default: includes rink coordinates + movement metrics
tracking_df = tracking_dict_to_df(frames)
print(f"Shape: {tracking_df.shape}")
print(f"Columns: {list(tracking_df.columns)}")

# extra=False: coordinates only, no speed columns
tracking_df_slim = tracking_dict_to_df(frames, extra=False)
print(f"\nSlim shape: {tracking_df_slim.shape}")

# Puck trajectory only
puck = tracking_df[tracking_df['is_puck']]
print(f"\nPuck frames: {len(puck)}")
puck[['timeStamp', 'rink_x', 'rink_y', 'speed']].head()


## Error Handling

ScraperNHL provides helpful error messages and handles common issues gracefully.

In [ ]:
# Example: Handling invalid inputs
from scrapernhl.exceptions import ScraperError, ValidationError

# Invalid league
try:
    data = scrape('invalid_league', 'stats', season=90)
except (ScraperError, KeyError) as e:
    print(f"Expected error for invalid league: {e}")

# Invalid data type
try:
    data = scrape('qmjhl', 'invalid_type', season=90)
except ValueError as e:
    print(f"Expected error for invalid data type: {e}")

# Missing required parameter
try:
    # pbp requires game_id
    data = scrape('qmjhl', 'pbp')
except (TypeError, ValueError) as e:
    print(f"Expected error for missing parameter: {e}")

## Performance Optimization

Tips for optimizing your scraping performance:

### 1. Use Pagination Wisely

Don't fetch all data at once - use `limit` and `offset`:

In [ ]:
# Good: Fetch only what you need
stats = scrape('ohl', 'stats', season=83, position='skaters', limit=20)
print(f"Fetched {len(stats)} players")

# Bad: Fetching too much data
# stats = scrape('ohl', 'stats', season=83, position='skaters', limit=1000)
# This is slower and uses more memory

### 2. Batch Requests Efficiently

When fetching multiple datasets, batch them intelligently:

In [ ]:
# Example: Fetch data for multiple teams
scraper = HockeyScraper('qmjhl')

team_ids = [1, 2, 3]  # Replace with actual team IDs
all_rosters = []

for team_id in team_ids:
    try:
        roster = scraper.roster(team_id=team_id, season=90)
        roster['team_id'] = team_id
        all_rosters.append(roster)
        print(f"Team {team_id}: {len(roster)} players")
    except Exception as e:
        print(f"Team {team_id}: Error - {e}")

if all_rosters:
    combined = pd.concat(all_rosters, ignore_index=True)
    print(f"\nTotal players: {len(combined)}")

### 3. Reuse Scraper Objects

Create one scraper object and reuse it:

In [ ]:
# Good: Reuse scraper
scraper = HockeyScraper('ohl')
stats1 = scraper.player_stats(season=83, position='skaters', limit=10)
stats2 = scraper.player_stats(season=83, position='goalies', limit=10)
schedule = scraper.schedule(season=83, team_id=-1)

print(f"Skaters: {len(stats1)}")
print(f"Goalies: {len(stats2)}")
print(f"Games: {len(schedule)}")

# Bad: Creating new scraper each time
# stats1 = HockeyScraper('ohl').player_stats(season=83, position='skaters')
# stats2 = HockeyScraper('ohl').player_stats(season=83, position='goalies')
# This creates unnecessary overhead

## Working with Large Datasets

Tips for handling large amounts of data:

In [ ]:
# Example: Process data in chunks
def fetch_all_stats(league, season, position, chunk_size=50):
    """Fetch all stats in chunks to avoid memory issues."""
    all_data = []
    offset = 0
    
    while True:
        chunk = scrape(league, 'stats', 
                      season=season, 
                      position=position, 
                      limit=chunk_size, 
                      offset=offset)
        
        if len(chunk) == 0:
            break
            
        all_data.append(chunk)
        offset += chunk_size
        
        print(f"Fetched {len(chunk)} records (total: {offset})")
        
        if len(chunk) < chunk_size:
            break
    
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

# Use it
# all_stats = fetch_all_stats('ohl', 83, 'skaters', chunk_size=50)
# print(f"Total players: {len(all_stats)}")

## Exporting to Different Formats

Advanced export options:

In [ ]:
# Get some data
stats = scrape('qmjhl', 'stats', season=90, position='skaters', limit=100)

# Export to CSV with compression
stats.to_csv('stats.csv.gz', compression='gzip', index=False)
print("Exported to compressed CSV")

# Export to Parquet (efficient binary format)
try:
    stats.to_parquet('stats.parquet', index=False)
    print("Exported to Parquet")
except ImportError:
    print("Install pyarrow for Parquet: pip install pyarrow")

# Export to JSON with custom formatting
stats.to_json('stats.json', orient='records', indent=2, force_ascii=False)
print("Exported to formatted JSON")

# Export to Excel with multiple sheets
try:
    with pd.ExcelWriter('hockey_data.xlsx') as writer:
        stats.to_excel(writer, sheet_name='Skaters', index=False)
        # Add more sheets as needed
    print("Exported to Excel")
except ImportError:
    print("Install openpyxl for Excel: pip install openpyxl")

## Custom Cache Configuration

You can work with the cache directly if needed:

In [ ]:
# Example: Working with cache
cache = Cache()

# Set a custom cache entry
cache.set('my_key', {'data': 'value'}, ttl=300)

# Get from cache
value = cache.get('my_key')
print(f"Cached value: {value}")

# Clear cache
# cache.clear()  # Clears all cached data

## Summary

This notebook covered:
- Built-in caching for performance
- Automatic rate limiting
- Data transformation pipeline
- Error handling best practices
- Performance optimization tips
- Working with large datasets
- Advanced export options

## Key Takeaways

1. **Caching is automatic** - No configuration needed
2. **Rate limiting is built-in** - Respects API limits automatically
3. **Reuse scraper objects** - More efficient than creating new ones
4. **Use pagination** - Don't fetch more data than you need
5. **Handle errors gracefully** - Use try/except for robust code
6. **Choose the right export format** - CSV for simple, Parquet for efficiency